In [ ]:
import threading
import numpy as np
from constants import ImageMetadata, Header, PIPE_PATH, NEG_MODE, SLICE_MODE
import json
import multiprocessing
from concurrent.futures import ThreadPoolExecutor

In [ ]:
def start_pipe_reader():
    """Read metadata and image bytes from the named pipe.

    Returns:
        tuple[ImageMetadata, Header]: Reconstructed metadata and processing
        header sent by the sender process.
    """
    print("Worker process started. Waiting for data...")

    # Open the named pipe and read the JSON header first, then raw image bytes.
    payload = {}
    data = b""
    with open(PIPE_PATH, "rb") as pipe:
        while True:
            try:
                metadata = pipe.readline()
                if not metadata:
                    # Pipe was closed by writer
                    break
                payload = json.loads(metadata.decode())
                colored = payload['header']['colored']
                if colored:
                    color = 3
                else:
                    color = 1
                # Compute exactly how many bytes to read from the stream.
                data = pipe.read(
                    payload["width"] * payload["height"] * color
                )
            except KeyboardInterrupt:
                break
            except Exception as e:
                print(f"Error: {e}")
                break

    # Rebuild typed objects used by the rest of the processing pipeline.
    metadata = ImageMetadata(payload["height"], payload["width"], payload["maxv"], data)
    header = Header(payload['header']['mode'], payload['header']['slice_range'], payload['header']['colored'])
    return metadata, header

In [ ]:
def image_negative_worker(start, end,result_lock, new_image, image):
    """Generate a negative for the image chunk of the image"""
    image_chunk = image[start:end].copy()
    new_image_chunk = 255 - image_chunk
    # Update only this thread's row range in the shared output array.
    with result_lock:
        new_image[start:end] += new_image_chunk
        
def threshold_with_slicing_worker(start, end,result_lock, new_image, image, limits, maxv):
    """Apply thresholding in a selected intensity range for a chunk."""
    a, b = limits
    image_chunk = image[start:end].copy()
    mask = (image_chunk > a) & (image_chunk < b)
    new_image_chunk = np.where(mask, maxv, image_chunk)
    
    # Write back the processed rows for this slice.
    with result_lock:
        new_image[start:end] = new_image_chunk

In [ ]:
def calculate_image_threads(height, min_rows_per_thread=50, max_threads=16):
    """
    Calculate threads for image processing

    Args:
        height: Image height in pixels
        min_rows_per_thread: Minimum rows to avoid overhead
        max_threads: Upper limit
    """
    cpu_cores = multiprocessing.cpu_count()

    # Maximum threads possible based on rows
    max_threads_by_rows = max(1, height // min_rows_per_thread)

    # Optimal threads (can't have more threads than rows)
    optimal = min(cpu_cores, max_threads_by_rows, max_threads)

    # Keep at least one CPU core free when all cores would be used
    if optimal == cpu_cores and optimal > 1:
        optimal -= 1

    # Ensure each thread gets reasonable work
    if optimal > 1:
        rows_per_thread = height // optimal
        if rows_per_thread < min_rows_per_thread:
            optimal = max(1, height // min_rows_per_thread)
            if optimal == cpu_cores and optimal > 1:
                optimal -= 1

    return optimal

In [ ]:
def get_image_slice_values(metadata, colored):
    """Build a numpy image array and row slices for thread execution.

    Args:
        metadata (ImageMetadata): Image dimensions, max value, and raw bytes.
        colored (bool): True for RGB image layout, False for grayscale.

    Returns:
        dict: Dictionary with parsed "image" ndarray and "slices" list.
    """
    threads = calculate_image_threads(metadata.height)

    width, height = metadata.width, metadata.height
    # Reconstruct numpy array from raw bytes based on image format.
    if colored:
        image = np.frombuffer(metadata.data, dtype=np.uint8).reshape(height, width, 3)
    else:
        image = np.frombuffer(metadata.data, dtype=np.uint8).reshape(height, width)
        
    rows_per_thread = height // threads

    # Split image rows into contiguous chunks for each worker thread.
    slices = []
    for i in range(threads):
        start = i * rows_per_thread
        end = (i + 1) * rows_per_thread if i < threads - 1 else height
        slices.append([start, end]) # slices

    return {
        "image": image,
        "slices": slices,
    }

In [ ]:
def generate_threads(image, slices, worker, args=None):
    """Execute a worker across row slices using a thread pool.

    Args:
        image (np.ndarray): Input image array.
        slices (list[list[int]]): Row intervals [start, end] per thread.
        worker (callable): Worker function to apply to each slice.
        args (list | None): Optional extra worker arguments.

    Returns:
        np.ndarray: Processed image with all worker results applied.
    """
    if args is None:
        args = []

    # Shared output buffer filled by all workers.
    new_image = np.zeros_like(image)
    lock = threading.Lock()

    # Dispatch one task per row slice and wait for all to complete.
    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        futures = [
            executor.submit(worker, start, end, lock, new_image, image, *args)
            for start, end in slices
        ]

        for future in futures:
            future.result()
    return new_image

In [ ]:
def process_image_with_worker(
    worker,
    metadata,
    output_path,
    colored,
    worker_args=None,
):
    """Run threaded image processing and write the resulting PGM/PPM file.

    Args:
        worker (callable): Function applied to each image slice.
        metadata (ImageMetadata): Input image metadata and bytes.
        output_path (str): Target path for the processed image file.
        colored (bool): Whether output should be encoded as PPM.
        worker_args (callable | list | None): Extra arguments for the worker.
    """
    image_values = get_image_slice_values(metadata, colored)
    image = image_values["image"]
    slices = image_values["slices"]

    # Support static args or a callback that derives args from metadata.
    args = worker_args(metadata) if callable(worker_args) else (worker_args or [])

    new_image = generate_threads(
        image,
        slices,
        worker,
        args,
    )

    altered_bytes = new_image.tobytes()
    if colored:
        header = f"P6\n{metadata.width} {metadata.height}\n255\n".encode()
    else:
        header = f"P5\n{metadata.width} {metadata.height}\n255\n".encode()

    # Write a valid PPM/PGM file with header + transformed bytes.
    with open(output_path, "wb") as f:
        f.write(header)
        f.write(altered_bytes)

In [ ]:
from pathlib import Path

def main():
    """Run worker flow: receive payload, process image, and write output."""
    # Receive image + operation metadata from sender process.
    metadata, header = start_pipe_reader()

    output_dir = Path("output")
    output_dir.mkdir(parents=True, exist_ok=True)

    ext = "ppm" if header.colored else "pgm"
    if header.mode == SLICE_MODE:
        # Run threshold mode with additional [slice_range, maxv] worker args.
        process_image_with_worker(
            threshold_with_slicing_worker,
            metadata,
            output_path=str(output_dir / f"output.{ext}"),
            colored=header.colored,
            worker_args=lambda metadata: [header.slice_range, metadata.maxv],
        )
    elif header.mode == NEG_MODE:
        # Run negative mode without extra worker args.
        process_image_with_worker(
            image_negative_worker,
            metadata,
            output_path=str(output_dir / f"output.{ext}"),
            colored=header.colored,
        )
    else:
        print("No mode selected!")

In [ ]:
if __name__ == "__main__":
    main()
    print("Worker process finished")